# Python Iterators and Generators

This notebook provides a comprehensive exploration of iterators and generators in Python, two powerful features that enable efficient iteration and lazy evaluation of data sequences.

## Table of Contents
1. [Understanding Iteration in Python](#understanding-iteration)
2. [The Iterator Protocol](#iterator-protocol)
3. [Creating Custom Iterators](#custom-iterators)
4. [Introduction to Generators](#intro-generators)
5. [Generator Functions with yield](#generator-functions)
6. [Generator Expressions](#generator-expressions)
7. [Memory Efficiency with Generators](#memory-efficiency)
8. [Lazy Evaluation](#lazy-evaluation)
9. [Advanced Generator Features](#advanced-features)
10. [Practical Applications and Examples](#applications)

<a id="understanding-iteration"></a>
## 1. Understanding Iteration in Python

Iteration is a fundamental concept in Python that allows us to traverse through a collection of items one at a time. Many Python objects are designed to be iterable, meaning they can be used in a `for` loop.

Python's iteration system is built around the concept of **iterables** - objects that can return their members one at a time.

In [ ]:
# Demonstrating iteration with different iterable types

# Lists
print("Iterating through a list:")
fruits = ["apple", "banana", "cherry", "date"]
for fruit in fruits:
    print(f"- {fruit}")

# Tuples
print("\nIterating through a tuple:")
colors = ("red", "green", "blue")
for color in colors:
    print(f"- {color}")

# Strings
print("\nIterating through a string:")
word = "Python"
for character in word:
    print(f"- {character}")

# Dictionaries (iterates through keys by default)
print("\nIterating through a dictionary:")
user = {"name": "John", "age": 30, "city": "New York"}
for key in user:
    print(f"- {key}: {user[key]}")

# Sets
print("\nIterating through a set:")
numbers = {1, 2, 3, 4, 5}
for num in numbers:
    print(f"- {num}")

### How Iteration Works Behind the Scenes

When you use a `for` loop in Python, several things happen:

1. The `for` statement calls `iter()` on the iterable object
2. The `iter()` function returns an iterator object
3. The `for` loop repeatedly calls `next()` on this iterator
4. When there are no more items, a `StopIteration` exception is raised, which the `for` loop handles to terminate the loop

Let's see this process explicitly:

In [ ]:
# Manual iteration demonstration
numbers = [1, 2, 3, 4, 5]

# Get an iterator from the list
iterator = iter(numbers)

# Manually iterate using next()
try:
    print(next(iterator))  # 1
    print(next(iterator))  # 2
    print(next(iterator))  # 3
    print(next(iterator))  # 4
    print(next(iterator))  # 5
    print(next(iterator))  # Will raise StopIteration
except StopIteration:
    print("Iteration complete!")

<a id="iterator-protocol"></a>
## 2. The Iterator Protocol

The iterator protocol consists of two methods that enable an object to be iterable:

1. `__iter__()`: Returns the iterator object itself. This is called by the built-in `iter()` function.
2. `__next__()`: Returns the next item in the sequence. When there are no more items, it raises a `StopIteration` exception.

An iterator is an object that implements both these methods, forming what's called the **iterator protocol**.

In [ ]:
# Demonstrating the iterator protocol with a simple list
simple_list = [1, 2, 3]

# Get the iterator
iterator = iter(simple_list)
print(f"Iterator object: {iterator}")

# The iterator has a __next__ method
print(f"First item: {iterator.__next__()}")
print(f"Second item: {next(iterator)}")  # next() is a built-in function that calls __next__()
print(f"Third item: {next(iterator)}")

# After the last item, StopIteration is raised
try:
    next(iterator)
except StopIteration:
    print("StopIteration exception raised!")

### Built-in Functions for Iteration

Python provides several built-in functions designed to work with iterables and iterators:

- `iter()`: Creates an iterator from an iterable
- `next()`: Retrieves the next item from an iterator
- `enumerate()`: Adds a counter to an iterable
- `zip()`: Combines multiple iterables
- `map()`: Applies a function to each item in an iterable
- `filter()`: Filters items based on a function

Let's see a few of these in action:

In [ ]:
# Using built-in functions with iterables
colors = ["red", "green", "blue", "yellow"]

# enumerate() - Get index and value pairs
print("Using enumerate():")
for index, color in enumerate(colors):
    print(f"{index}: {color}")

# zip() - Combining iterables
print("\nUsing zip():")
sizes = ["small", "medium", "large", "xl"]
for color, size in zip(colors, sizes):
    print(f"{color} - {size}")

# map() - Apply a function to each item
print("\nUsing map():")
lengths = list(map(len, colors))
print(f"Lengths of colors: {lengths}")

# filter() - Keep only items that match a condition
print("\nUsing filter():")
long_colors = list(filter(lambda x: len(x) > 4, colors))
print(f"Colors with length > 4: {long_colors}")

<a id="custom-iterators"></a>
## 3. Creating Custom Iterators

To create a custom iterator, we need to implement the iterator protocol in a class. This means implementing both `__iter__()` and `__next__()` methods.

Here's an example of a custom iterator that generates a sequence of even numbers:

In [ ]:
class EvenNumbers:
    """Iterator that generates even numbers up to a max value"""
    
    def __init__(self, max_value):
        self.max_value = max_value
        self.current = 0
    
    def __iter__(self):
        """Return the iterator object itself"""
        return self
    
    def __next__(self):
        """Return the next even number or raise StopIteration"""
        if self.current > self.max_value:
            raise StopIteration
        
        result = self.current
        self.current += 2
        return result

# Using our custom iterator
even_nums = EvenNumbers(10)
for num in even_nums:
    print(num)

# Once exhausted, the iterator is done
print("\nTrying to iterate again:")
for num in even_nums:
    print(num)  # Nothing will be printed as the iterator is exhausted

If we want to use our iterator multiple times, we need to separate the iterable and iterator:

In [ ]:
class EvenNumbersIterable:
    """An iterable that generates even numbers"""
    
    def __init__(self, max_value):
        self.max_value = max_value
    
    def __iter__(self):
        """Return a fresh iterator each time"""
        return EvenNumbersIterator(self.max_value)

class EvenNumbersIterator:
    """An iterator that generates even numbers"""
    
    def __init__(self, max_value):
        self.max_value = max_value
        self.current = 0
    
    def __next__(self):
        if self.current > self.max_value:
            raise StopIteration
        
        result = self.current
        self.current += 2
        return result

# Now we can reuse our iterable
evens = EvenNumbersIterable(10)

print("First iteration:")
for num in evens:
    print(num)

print("\nSecond iteration:")
for num in evens:
    print(num)  # We get the numbers again

### Practical Example: Fibonacci Iterator

Let's implement a Fibonacci sequence iterator as a more practical example:

In [ ]:
class Fibonacci:
    """Iterator that generates Fibonacci numbers up to n terms"""
    
    def __init__(self, n):
        self.n = n
        self.count = 0
        self.a, self.b = 0, 1
    
    def __iter__(self):
        return self
    
    def __next__(self):
        if self.count >= self.n:
            raise StopIteration
        
        result = self.a
        self.a, self.b = self.b, self.a + self.b
        self.count += 1
        return result

# Generate the first 10 Fibonacci numbers
fib = Fibonacci(10)
print("Fibonacci sequence:")
for num in fib:
    print(num, end=" ")

<a id="intro-generators"></a>
## 4. Introduction to Generators

Generators provide a simpler way to create iterators. Instead of creating a class and implementing the iterator protocol, you can define a function that uses the `yield` keyword to return values one at a time.

Generators are a special type of iterator that are defined with a function notation. When a generator function is called, it returns a generator object, which is an iterator.

Key differences between iterators and generators:

1. **Syntax**: Generators use functions with `yield`, while iterators use classes with `__iter__` and `__next__`.
2. **State Management**: Generators automatically save their state between yields, while iterators need to store state in instance variables.
3. **Simplicity**: Generators are more concise and easier to write.
4. **Memory Efficiency**: Both are memory efficient, but generators often have cleaner syntax for this purpose.

In [ ]:
# A simple generator function
def count_up_to(max_value):
    count = 1
    while count <= max_value:
        yield count
        count += 1

# Using the generator
counter = count_up_to(5)

# Generators are iterators
print(f"Generator object: {counter}")
print(f"First value: {next(counter)}")
print(f"Second value: {next(counter)}")
print(f"Third value: {next(counter)}")

# We can also use it in a for loop
print("\nRestarting with a new generator:")
for num in count_up_to(5):
    print(num, end=" ")

### Rewriting Our Examples as Generators

Let's rewrite our earlier examples using generator functions:

In [ ]:
# Even numbers generator
def even_numbers(max_value):
    current = 0
    while current <= max_value:
        yield current
        current += 2

# Fibonacci generator
def fibonacci(n):
    a, b = 0, 1
    count = 0
    while count < n:
        yield a
        a, b = b, a + b
        count += 1

# Using the generators
print("Even numbers:")
for num in even_numbers(10):
    print(num, end=" ")

print("\n\nFibonacci sequence:")
for num in fibonacci(10):
    print(num, end=" ")

<a id="generator-functions"></a>
## 5. Generator Functions with yield

The `yield` statement is what makes generator functions special. When a function contains a `yield` statement:

1. The function returns a generator object when called
2. The function's state is suspended when `yield` is encountered
3. The function resumes from where it left off when the generator's `__next__()` method is called
4. Local variables and their values are saved between calls

This allows generator functions to maintain their state between calls, which is a powerful feature for iteration.

In [ ]:
def demonstrate_yield():
    print("Starting the generator")
    yield 1
    print("After first yield")
    yield 2
    print("After second yield")
    yield 3
    print("After third yield")
    print("Generator exhausted")

# Using the generator
gen = demonstrate_yield()
print("Generator created, nothing executed yet")

print("\nGetting first value:")
value = next(gen)
print(f"Received: {value}")

print("\nGetting second value:")
value = next(gen)
print(f"Received: {value}")

print("\nGetting third value:")
value = next(gen)
print(f"Received: {value}")

try:
    print("\nTrying to get another value:")
    next(gen)
except StopIteration:
    print("StopIteration exception caught - generator is exhausted")

### Multiple yield Statements

Generator functions can have multiple `yield` statements, creating complex iteration patterns:

In [ ]:
def odd_even_generator(n):
    """Generate first odd then even numbers up to n"""
    # First yield all odd numbers
    for i in range(1, n + 1, 2):
        yield i
    
    # Then yield all even numbers
    for i in range(2, n + 1, 2):
        yield i

print("Numbers in odd-then-even order:")
for num in odd_even_generator(10):
    print(num, end=" ")

### yield vs. return

The key difference between `yield` and `return`:

1. `return` terminates the function and returns a value
2. `yield` pauses the function, saves its state, and returns a value, but allows the function to resume later

We can combine both in a generator function:

In [ ]:
def limited_generator(max_value, limit):
    """Generate numbers up to max_value but stop at limit"""
    for i in range(max_value):
        if i >= limit:
            return  # Terminate the generator early
        yield i

print("Limited generator (max=10, limit=5):")
for num in limited_generator(10, 5):
    print(num, end=" ")

print("\n\nLimited generator (max=10, limit=15):")
for num in limited_generator(10, 15):
    print(num, end=" ")  # Will print all numbers since limit > max_value

<a id="generator-expressions"></a>
## 6. Generator Expressions

Generator expressions are similar to list comprehensions but create generators instead of lists. They use parentheses instead of square brackets:

- List comprehension: `[expression for item in iterable]`
- Generator expression: `(expression for item in iterable)`

Generator expressions are more memory-efficient than list comprehensions since they don't create the entire sequence in memory at once.

In [ ]:
import sys

# Compare list comprehension vs generator expression
numbers = range(1000000)

# List comprehension
list_comp = [x * 2 for x in numbers]
list_size = sys.getsizeof(list_comp)

# Generator expression
gen_exp = (x * 2 for x in numbers)
gen_size = sys.getsizeof(gen_exp)

print(f"List comprehension size: {list_size:,} bytes")
print(f"Generator expression size: {gen_size:,} bytes")
print(f"Memory saved: {(list_size - gen_size):,} bytes")

# Using a generator expression
print("\nFirst 10 values from generator expression:")
count = 0
for num in gen_exp:
    if count < 10:
        print(num, end=" ")
        count += 1
    else:
        break

### Complex Generator Expressions

Just like list comprehensions, generator expressions can use conditions and nested loops:

In [ ]:
# Generator expression with condition
even_squares = (x * x for x in range(20) if x % 2 == 0)
print("Even squares:")
for square in even_squares:
    print(square, end=" ")

# Generator expression with nested loops - creating pairs
pairs = ((x, y) for x in range(3) for y in range(3))
print("\n\nCoordinate pairs:")
for pair in pairs:
    print(pair, end=" ")

# Chain multiple generator expressions using the built-in itertools
import itertools

gen1 = (x for x in range(5))
gen2 = (x * 10 for x in range(5))
combined = itertools.chain(gen1, gen2)

print("\n\nChained generators:")
for num in combined:
    print(num, end=" ")

<a id="memory-efficiency"></a>
## 7. Memory Efficiency with Generators

One of the main advantages of generators is memory efficiency. Instead of creating the entire sequence in memory, generators produce values one at a time.

This is especially useful when working with large datasets or infinite sequences.

In [ ]:
import time
import sys

# Define a function that reads a large file line by line using a list
def read_file_list(filename):
    with open(filename, 'r') as f:
        return f.readlines()

# Define a function that reads a large file line by line using a generator
def read_file_generator(filename):
    with open(filename, 'r') as f:
        for line in f:
            yield line

# Create a large sample file for demonstration
def create_sample_file(filename, lines=100000):
    with open(filename, 'w') as f:
        for i in range(lines):
            f.write(f"This is line {i} of the test file.\n")
    return filename

# Create our test file
test_file = create_sample_file('large_test_file.txt')

# Compare memory usage
print("Comparing memory usage:")
print("-" * 40)

# Using list
start_time = time.time()
lines_list = read_file_list(test_file)
list_time = time.time() - start_time
list_memory = sys.getsizeof(lines_list)
print(f"List method:")
print(f"  Time: {list_time:.4f} seconds")
print(f"  Memory: {list_memory:,} bytes")
print(f"  First 5 lines: {lines_list[:5]}")

# Using generator
start_time = time.time()
lines_gen = read_file_generator(test_file)
gen_time = time.time() - start_time
gen_memory = sys.getsizeof(lines_gen)
print(f"\nGenerator method:")
print(f"  Time: {gen_time:.4f} seconds")
print(f"  Memory: {gen_memory:,} bytes")

# Get first 5 lines from generator
first_5_gen = []
for i, line in enumerate(lines_gen):
    if i < 5:
        first_5_gen.append(line)
    else:
        break
print(f"  First 5 lines: {first_5_gen}")

print(f"\nMemory savings: {list_memory - gen_memory:,} bytes")
print(f"Memory reduction: {(1 - gen_memory/list_memory) * 100:.2f}%")

# Clean up our test file
import os
os.remove(test_file)

### Processing Large Datasets with Generators

When working with data science or big data applications, generators can be extremely useful for processing large datasets. Here's a simple example:

In [ ]:
# Define a generator for processing data in chunks
def process_in_chunks(data, chunk_size=100):
    for i in range(0, len(data), chunk_size):
        chunk = data[i:i + chunk_size]
        # Process the chunk
        result = sum(chunk)
        yield result

# Generate some sample data
large_data = list(range(10000))

# Process the data in chunks
chunk_results = list(process_in_chunks(large_data, 1000))
print(f"Number of chunks processed: {len(chunk_results)}")
print(f"Sum of each chunk: {chunk_results}")
print(f"Total sum: {sum(chunk_results)}")

<a id="lazy-evaluation"></a>
## 8. Lazy Evaluation

Generators implement lazy evaluation, which means values are computed only when needed. This allows working with:

1. Infinite sequences
2. Sequences that would be too large to fit in memory
3. Data streams where values are not all available at once

This on-demand computation is more efficient and enables certain algorithms that wouldn't be possible with eager evaluation.

In [ ]:
# A generator representing an infinite sequence of powers of 2
def powers_of_two():
    n = 0
    while True:  # This is an infinite loop!
        yield 2 ** n
        n += 1

# Using the infinite generator safely
power_gen = powers_of_two()

print("First 10 powers of 2:")
for i in range(10):
    print(next(power_gen), end=" ")

# Another example - Fibonacci sequence without upper limit
def fibonacci_infinite():
    a, b = 0, 1
    while True:
        yield a
        a, b = b, a + b

print("\n\nFirst 15 Fibonacci numbers:")
fib_gen = fibonacci_infinite()
for i in range(15):
    print(next(fib_gen), end=" ")

### Combining Generators with Filtering

Generators work well with filtering operations, maintaining lazy evaluation:

In [ ]:
# Generate prime numbers using lazy evaluation
def is_prime(n):
    """Check if a number is prime"""
    if n <= 1:
        return False
    if n <= 3:
        return True
    if n % 2 == 0 or n % 3 == 0:
        return False
    
    i = 5
    while i * i <= n:
        if n % i == 0 or n % (i + 2) == 0:
            return False
        i += 6
    return True

def primes_up_to(limit):
    """Generate prime numbers up to the limit"""
    for num in range(2, limit + 1):
        if is_prime(num):
            yield num

# Generate primes up to 100
print("Prime numbers up to 100:")
for prime in primes_up_to(100):
    print(prime, end=" ")

<a id="advanced-features"></a>
## 9. Advanced Generator Features

Python generators have advanced features beyond simple iteration:

1. **send()**: Send a value into the generator
2. **throw()**: Raise an exception inside the generator
3. **close()**: Close the generator

These methods allow for two-way communication with generators, making them suitable for complex control flows and coroutines.

In [ ]:
# A generator that can receive values with send()
def echo_generator():
    value = yield "Ready for input"
    while True:
        value = yield f"You sent: {value}"

# Create and initialize the generator
echo = echo_generator()
initial = next(echo)  # Prime the generator
print(f"Generator says: {initial}")

# Send values to the generator
responses = []
for message in ["Hello", "How are you?", "Goodbye"]:
    print(f"Sending: '{message}'")
    response = echo.send(message)
    print(f"Generator responds: '{response}'")
    responses.append(response)

print(f"\nAll responses: {responses}")

### The throw() Method

The `throw()` method allows you to inject exceptions into the generator:

In [ ]:
# A generator that can handle exceptions
def exception_handling_generator():
    try:
        yield "First value"
        yield "Second value"
        yield "Third value"
    except ValueError:
        yield "Caught ValueError!"
    except Exception as e:
        yield f"Caught other exception: {type(e).__name__}"
    finally:
        yield "In finally block"

# Create the generator
gen = exception_handling_generator()
print(next(gen))  # "First value"

# Throw a ValueError into the generator
print(gen.throw(ValueError))  # "Caught ValueError!"

# Continue with the generator
print(next(gen))  # "In finally block"

# Create a new generator to demonstrate other exceptions
gen = exception_handling_generator()
print(next(gen))  # "First value"
try:
    print(gen.throw(RuntimeError))  # "Caught other exception: RuntimeError"
    print(next(gen))  # "In finally block"
except StopIteration:
    print("Generator exhausted")

### The close() Method

The `close()` method allows you to terminate a generator:

In [ ]:
# A generator that demonstrates the close() method
def closeable_generator():
    try:
        yield "First value"
        yield "Second value"
        yield "Third value"
    except GeneratorExit:
        print("Generator is being closed!")
    finally:
        print("Cleanup in finally block")

# Create and use the generator
gen = closeable_generator()
print(next(gen))  # "First value"
print(next(gen))  # "Second value"

# Close the generator
gen.close()

# Try to use the generator after closing
try:
    next(gen)
except StopIteration:
    print("Generator is closed and cannot produce more values")

### Building Pipelines with Generators

One powerful application of generators is building data processing pipelines:

In [ ]:
# A series of generator functions to create a processing pipeline

def read_data():
    """Simulate reading data from a source"""
    data = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
    for item in data:
        yield item

def filter_even(items):
    """Filter out odd numbers"""
    for item in items:
        if item % 2 == 0:
            yield item

def multiply_by_three(items):
    """Multiply each item by 3"""
    for item in items:
        yield item * 3

def convert_to_string(items):
    """Convert numbers to strings"""
    for item in items:
        yield f"Item: {item}"

# Create the pipeline
def create_pipeline():
    data = read_data()
    filtered_data = filter_even(data)
    multiplied_data = multiply_by_three(filtered_data)
    result = convert_to_string(multiplied_data)
    return result

# Execute the pipeline
pipeline = create_pipeline()
print("Pipeline results:")
for result in pipeline:
    print(result)

<a id="applications"></a>
## 10. Practical Applications and Examples

Iterators and generators have many practical applications in real-world Python programming. Here are some examples:

### 1. File Processing

Processing large files line by line without loading the entire file into memory:

In [ ]:
# Process a file line by line
def grep(file_path, search_string):
    """A simple grep-like function that yields matching lines from a file"""
    with open(file_path, 'w') as f:
        # Create a sample file first
        f.write("This is line 1\n")
        f.write("This contains the word python\n")
        f.write("Python is great for data processing\n")
        f.write("This is the final line\n")
    
    with open(file_path, 'r') as f:
        line_num = 0
        for line in f:
            line_num += 1
            if search_string.lower() in line.lower():
                yield f"Line {line_num}: {line.strip()}"

# Use the generator to search for lines containing "python"
print("Lines containing 'python':")
for match in grep('sample_text.txt', 'python'):
    print(match)

# Clean up
import os
os.remove('sample_text.txt')

### 2. Data Processing Pipeline

Creating a data processing pipeline for a simple ETL (Extract, Transform, Load) process:

In [ ]:
# Simple ETL pipeline using generators
import csv
import os

# Create sample data
def create_sample_csv():
    with open('sales_data.csv', 'w', newline='') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(['date', 'product', 'quantity', 'unit_price'])
        writer.writerow(['2023-01-15', 'laptop', '5', '999.99'])
        writer.writerow(['2023-01-16', 'monitor', '10', '249.50'])
        writer.writerow(['2023-01-17', 'keyboard', '15', '49.99'])
        writer.writerow(['2023-01-18', 'mouse', '20', '24.99'])
        writer.writerow(['2023-01-19', 'laptop', '3', '999.99'])
    return 'sales_data.csv'

# Extract: Read data from CSV
def extract_from_csv(file_path):
    with open(file_path, 'r') as csvfile:
        reader = csv.DictReader(csvfile)
        for row in reader:
            yield row

# Transform: Calculate total price and convert types
def transform_data(data_rows):
    for row in data_rows:
        row['quantity'] = int(row['quantity'])
        row['unit_price'] = float(row['unit_price'])
        row['total_price'] = row['quantity'] * row['unit_price']
        yield row

# Load: Calculate summary statistics
def calculate_stats(transformed_rows):
    products = {}
    total_sales = 0
    
    # Process each row
    for row in transformed_rows:
        product = row['product']
        total = row['total_price']
        
        # Update product stats
        if product not in products:
            products[product] = {'units': 0, 'revenue': 0}
        
        products[product]['units'] += row['quantity']
        products[product]['revenue'] += total
        total_sales += total
        
        # Yield the row for further processing if needed
        yield row
    
    # After processing all rows, print summary
    print("\nSummary Statistics:")
    print(f"Total Sales: ${total_sales:.2f}")
    print("\nProduct Breakdown:")
    for product, stats in products.items():
        print(f"{product}: {stats['units']} units, ${stats['revenue']:.2f} revenue")

# Run the ETL pipeline
def run_pipeline():
    file_path = create_sample_csv()
    
    print("Starting ETL pipeline...\n")
    
    # Create the pipeline
    extracted_data = extract_from_csv(file_path)
    transformed_data = transform_data(extracted_data)
    
    # Process and collect the results
    print("Processed Records:")
    print("------------------")
    for row in calculate_stats(transformed_data):
        print(f"- {row['date']}: {row['quantity']} {row['product']}(s) at ${row['unit_price']:.2f} each = ${row['total_price']:.2f}")
    
    # Clean up
    os.remove(file_path)

# Execute the pipeline
run_pipeline()

### 3. Working with Infinite Sequences

Generators are perfect for working with potentially infinite sequences:

In [ ]:
import itertools

# Generate an infinite sequence of incrementing ids
def id_generator(start=1):
    """Generate infinite sequence of IDs"""
    counter = start
    while True:
        yield f"ID-{counter:04d}"
        counter += 1

# Create a database record generator
def record_generator(id_gen, num_records=10):
    """Generate sample database records"""
    for i in range(num_records):
        record_id = next(id_gen)
        yield {
            'id': record_id,
            'name': f"User {i+1}",
            'active': i % 3 != 0  # Every 3rd user is inactive
        }

# Generate IDs starting from 1000
id_gen = id_generator(1000)

# Generate 10 records
print("Generated Records:")
for record in record_generator(id_gen, 10):
    print(record)

# We can continue with the same ID sequence later
print("\nGenerated 5 more records:")
for record in record_generator(id_gen, 5):
    print(record)

# Using itertools to work with infinite sequences
print("\nFirst 5 items from an infinite cycle:")
cycle = itertools.cycle(["A", "B", "C"])
for i, item in enumerate(cycle):
    if i >= 5:
        break
    print(item, end=" ")

### 4. Memory-Efficient Data Analysis

Using generators for memory-efficient data analysis with large datasets:

In [ ]:
import random

# Generate a large dataset of temperature readings
def temperature_readings(days=365, readings_per_day=24):
    """Generate simulated temperature readings"""
    for day in range(1, days + 1):
        for hour in range(readings_per_day):
            # Simulate temperature with daily and seasonal variations
            base_temp = 15 + 10 * random.random()  # Base temperature between 15-25°C
            daily_variation = 5 * math.sin(hour * math.pi / 12)  # Daily cycle
            seasonal_variation = 10 * math.sin(day * 2 * math.pi / 365)  # Yearly cycle
            noise = random.normalvariate(0, 1)  # Random noise
            
            temperature = base_temp + daily_variation + seasonal_variation + noise
            yield (day, hour, temperature)

import math

# Analyze the data without storing all readings in memory
def analyze_temperatures(readings):
    """Calculate statistics from temperature readings generator"""
    count = 0
    temp_sum = 0
    temp_min = float('inf')
    temp_max = float('-inf')
    
    # Process one reading at a time
    for day, hour, temp in readings:
        count += 1
        temp_sum += temp
        temp_min = min(temp_min, temp)
        temp_max = max(temp_max, temp)
        
        # Report progress periodically
        if count % 1000 == 0:
            print(f"Processed {count} readings...")
    
    if count > 0:
        avg_temp = temp_sum / count
        return {
            'count': count,
            'avg_temp': avg_temp,
            'min_temp': temp_min,
            'max_temp': temp_max
        }
    return None

# Generate a year of hourly temperature readings (365 * 24 = 8,760 readings)
readings = temperature_readings(days=365, readings_per_day=24)

# Analyze the data
print("Analyzing temperature data...")
stats = analyze_temperatures(readings)
print("\nResults:")
print(f"Total readings: {stats['count']:,}")
print(f"Average temperature: {stats['avg_temp']:.2f}°C")
print(f"Minimum temperature: {stats['min_temp']:.2f}°C")
print(f"Maximum temperature: {stats['max_temp']:.2f}°C")

## Summary

Iterators and generators are powerful features in Python that provide memory-efficient and elegant solutions for handling sequences of data. Here's a recap of what we've covered:

1. **Iterators** are objects that implement the iterator protocol (`__iter__` and `__next__` methods).
2. **Generators** are special iterators created using functions with the `yield` statement.
3. **Generator expressions** provide a concise way to create generators, similar to list comprehensions.
4. **Lazy evaluation** allows for working with large or infinite sequences efficiently.
5. **Advanced generator features** like `send()`, `throw()`, and `close()` enable complex control flows.
6. **Practical applications** include file processing, data pipelines, and memory-efficient data analysis.

These tools are essential for writing Pythonic, memory-efficient code and are widely used in data processing, web development, and system programming.